In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms
import numpy as np
import os, time

In [ ]:
#-----------------------------------------------------------------------------#
#    IMAGE PREPROCESSING                                                      #
#-----------------------------------------------------------------------------#
## DIRECTORIES with datasets
img_path = 'dataset'

# Dataset with Transformation (using only 100 images)
full_dataset = datasets.ImageFolder(
    img_path,
    transforms.Compose([
        transforms.Resize((128, 256)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.0, 0.0, 0.0], 
            std=[1.0, 1.0, 1.0])
    ])
)
dataset = torch.utils.data.Subset(full_dataset, list(range(1)))

print(f"Dataset size: {len(dataset)}")

In [ ]:
#-----------------------------------------------------------------------------#
#    DATA LOADER                                                              #
#-----------------------------------------------------------------------------#
dataloader = DataLoader(dataset , batch_size=1, shuffle=False, num_workers=1)

# print(dataset)
# print(dataloader)
print("Min/Max:", dataset[0][0].min(), dataset[0][0].max())

# Test CPU

In [ ]:
class vaemodel1(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),  # Increased channels, stride 2
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.mu = nn.Linear(256, 6)  # Reduced latent space size
        self.std = nn.Linear(256, 6)  # Reduced latent space size

    def forward(self, x):
        a = self.encoder(x).permute(0,2,3,1)
        a = torch.flatten(a, start_dim=1)
        mu = self.mu(a)
        lvar = self.std(a)
        out = torch.cat((mu, lvar), dim=1)
        return out

model = vaemodel1()

In [ ]:
model.load_state_dict(torch.load("pre_trained_w_encoder.pt", weights_only=True))

In [ ]:
model.eval()
with torch.no_grad():
    inference_time = 0
    cpu_z = []
    for image, _ in dataloader:
        # Run inference for each image individually
        start_time = time.time()
        pred = model(image)
        inference_time += time.time() - start_time
        cpu_z.append(pred)

    # Convert list to numpy array
    cpu_z = np.array(cpu_z)

    # calculate average inference time and FPS
    num_images = len(cpu_z)
    avg_inference_time = inference_time / num_images
    fps = num_images / inference_time

    print(f"The total inference time was {inference_time:.6f} seconds for {num_images}.")
    print("Average inference time per image: {:.6f} seconds".format(avg_inference_time))
    print("FPS: {:.2f}".format(fps))

# Test DPU

In [ ]:
from pynq_dpu import DpuOverlay
overlay = DpuOverlay("vitisai_bitstream/dpu.bit")

In [ ]:
overlay.load_model("zcu104_vaemodel1.xmodel")

In [ ]:
dpu = overlay.runner

inputTensors = dpu.get_input_tensors()
outputTensors = dpu.get_output_tensors()

shapeIn = tuple(inputTensors[0].dims)
shapeOut = tuple(outputTensors[0].dims)
outputSize = int(outputTensors[0].get_data_size() / shapeIn[0])

output_data = [np.empty(shapeOut, dtype=np.float32, order="C")]
input_data = [np.empty(shapeIn, dtype=np.float32, order="C")]
data = input_data[0]

In [ ]:
inference_time = 0
vitisAI_z = []
for image, _ in dataloader:
    data = image.permute(0,2,3,1)
    start_time = time.time()
    job_id = dpu.execute_async(input_data, output_data)
    dpu.wait(job_id)
    inference_time += time.time() - start_time
    #o_data = output_data[0].reshape(2, 6)
    #std = np.exp(o_data[1]*0.5)
    #eps = np.random.randn(*std.shape)
    #vitisAI_z.append(o_data[0]+eps*std)
    vitisAI_z.append(output_data[0])

# Convert list to numpy array
vitisAI_z = np.array(vitisAI_z)

num_images = len(vitisAI_z)
avg_infer_time = inference_time/num_images
fps = num_images/inference_time
print(f"The total inference time was {inference_time:.6f} seconds for {num_images}.")
print("Average inference time: {} s".format(avg_infer_time))
print("Performance: {} FPS".format(fps))

In [ ]:
# Calculate MSE
mse = np.mean((cpu_z - vitisAI_z) ** 2)
print(f"Mean Squared Error between CPU and VitisAI outputs: {mse:.6f}")
print(cpu_z)
print(vitisAI_z)

# Test FINN

In [ ]:
from finn_driver import FINNExampleOverlay
from pynq.pl_server.device import Device
from qonnx.core.datatype import DataType

# dictionary describing the I/O of the FINN-generated accelerator
io_shape_dict = {
    # FINN DataType for input and output tensors
    "idt" : [DataType['INT8']],
    "odt" : [DataType['INT24']],
    # shapes for input and output tensors (NHWC layout)
    "ishape_normal" : [(1, 128, 256, 3)],
    "oshape_normal" : [(1, 64, 128, 16)],
    # folded / packed shapes below depend on idt/odt and input/output
    # PE/SIMD parallelization settings -- these are calculated by the
    # FINN compiler.
    "ishape_folded" : [(1, 128, 256, 1, 3)],
    "oshape_folded" : [(1, 64, 128, 1, 16)],
    "ishape_packed" : [(1, 128, 256, 1, 3)],
    "oshape_packed" : [(1, 64, 128, 1, 48)],
    "input_dma_name" : ['idma0'],
    "output_dma_name" : ['odma0'],
    "number_of_external_weights": 0,
    "num_inputs" : 1,
    "num_outputs" : 1,
}

device = Device.devices[0]
# instantiate FINN accelerator driver and pass batchsize and bitfile
accel = FINNExampleOverlay(
    bitfile_name = "finn_bitstream/finn-accel.bit", platform = "zynq-iodma",
    io_shape_dict = io_shape_dict, batch_size=1,
    runtime_weight_dir = "runtime_weights/", device=device
)

In [ ]:
print(accel.throughput_test())

In [ ]:
inference_time = 0
finn_z = []
for image, _ in dataloader:
    #in_data = image.permute(0,2,3,1)
    in_data = np.ones((1, 128, 256, 3), dtype=np.int8)
    start_t = time.time()
    out_data = accel.execute(in_data)
    inference_time += time.time() - start_t
    finn_z.append(out_data)

num_images = len(finn_z)
avg_infer_time = inference_time/num_images
fps = num_images/inference_time
print(f"The total inference time was {inference_time:.6f} seconds for {num_images}.")
print("Average inference time: {} s".format(avg_infer_time))
print("Performance: {} FPS".format(fps))

# Test HLS

In [ ]:
from vae_hls_driver import VAEOverlay
overlay = VAEOverlay("hls_bitstream/design_1.bit")

In [ ]:
inference_time = 0
hls_z = []
for image, _ in dataloader:
    in_data = image.permute(0,2,3,1)
    start_time = time.time()
    out_data = overlay.run(in_data)
    inference_time += time.time() - start_time
    hls_z.append(out_data)

# Convert list to numpy array
hls_z = np.array(hls_z)

num_images = len(hls_z)
avg_infer_time = inference_time/num_images
fps = num_images/inference_time
print(f"The total inference time was {inference_time:.6f} seconds for {num_images}.")
print("Average inference time: {} s".format(avg_infer_time))
print("Performance: {} FPS".format(fps))